In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import LoraConfig, get_peft_model
from trl import GRPOConfig, GRPOTrainer

/opt/conda/envs/trl/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
DATASET_NAME = "rmanluo/RoG-cwq"

TEXT_COLUMN = "prompt"
LABEL_COLUMN = "answer"

MAX_LENGTH = 1024


In [6]:
dataset = load_dataset(DATASET_NAME)

train_dataset = dataset["train"]
eval_dataset = dataset.get("validation", None)


In [7]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    padding_side="right",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    load_in_4bit=True,
    device_map="auto",
)

model.config.use_cache = False


`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
def reward_fn(prompts, generations, references, **kwargs):
    rewards = []

    for gen, ref in zip(generations, references):
        gen = gen.strip().lower()
        ref = ref.strip().lower()

        reward = 0.0

        if ref in gen:
            reward += 1.0

        # discourage empty / extremely short answers
        reward -= 0.001 * abs(len(gen) - len(ref))

        rewards.append(reward)

    return rewards


In [ ]:
grpo_config = GRPOConfig(
    output_dir="./grpo-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,

    num_generations=4,
    max_prompt_length=512,
    max_completion_length=512,

    beta=0.1,

    logging_steps=10,
    save_steps=500,
    save_total_limit=2,

    bf16=True,
    report_to="none",
)


In [ ]:
trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    args=grpo_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    reward_fn=reward_fn,
    prompt_column=TEXT_COLUMN,
    response_column=LABEL_COLUMN,
)


## Inject refactx

In [ ]:
trainer.generation_kwargs = {
    "max_new_tokens": grpo_config.max_completion_length,
    "do_sample": True,
    "temperature": 1.0,
    "top_p": 0.95,
    "logits_processor": logits_processor,
}

In [ ]:
trainer.train()


In [ ]:
trainer.model.save_pretrained("./grpo-lora-adapters")
tokenizer.save_pretrained("./grpo-lora-adapters")


# Judge

In [ ]:
JUDGE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

judge_model.eval()
for p in judge_model.parameters():
    p.requires_grad = False


In [ ]:
def build_judge_prompt(question, answer, facts):
    return f"""
You are a strict evaluator for question answering.

Question:
{question}

Model answer:
{answer}

Supporting facts:
{facts}

Evaluate the answer and return ONLY a JSON object with the following keys:
- supported: 1 if the answer is fully supported by the facts, else 0
- complete: 1 if the answer addresses all parts of the question, else 0
- correct: 1 if the answer is factually correct, else 0

Return only valid JSON.
"""


In [ ]:
import json

@torch.no_grad()
def run_judge(prompt, max_new_tokens=128):
    inputs = judge_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
    ).to(judge_model.device)

    outputs = judge_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        pad_token_id=judge_tokenizer.eos_token_id,
    )

    text = judge_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

    try:
        return json.loads(text)
    except Exception:
        return {"supported": 0, "complete": 0, "correct": 0}


In [ ]:
def llm_judge_reward(
    prompts,
    generations,
    references=None,
    facts=None,
    **kwargs,
):
    rewards = []

    for i in range(len(generations)):
        judge_prompt = build_judge_prompt(
            question=prompts[i],
            answer=generations[i],
            facts=facts[i],
        )

        scores = run_judge(judge_prompt)

        # weighted sum (tune freely)
        reward = (
            1.0 * scores["correct"]
            + 0.5 * scores["supported"]
            + 0.5 * scores["complete"]
        )

        rewards.append(float(reward))

    return rewards


In [ ]:
trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    args=grpo_config,
    train_dataset=train_dataset,
    reward_fn=llm_judge_reward,
    prompt_column="prompt",
    response_column="answer",
    additional_columns=["facts"],
)


In [ ]:
trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    args=grpo_config,
    train_dataset=train_dataset,
    reward_fns=[
        llm_judge_reward,
        length_penalty_reward,
    ],
    reward_weights=[1.0, 0.1],
)
